# Notebook 05 — Final Load Prep
**Project:** Ola Bengaluru Rides — Cancellation Intelligence & Revenue Recovery  
**Dataset:** Bengaluru Ola Rides, January 2024  
**Author:** Joshit  
**Institute:** Newton School of Technology — DVA Capstone 2

Last notebook before Tableau. Computing all KPIs and preparing 3 separate CSV files — one for each dashboard. Tableau will connect directly to these files.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('ready.')

In [ ]:
# update path if running locally: '../data/processed/ola_bengaluru_cleaned.csv'
CLEAN_PATH = '/content/ola_bengaluru_cleaned.csv'

df = pd.read_csv(CLEAN_PATH)
df['date'] = pd.to_datetime(df['date'])
df_success = df[df['is_successful'] == 1].copy()

print(f'loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'successful rides: {len(df_success):,}')

### Computing master KPIs
These will be used across all 3 dashboards as top-row KPI cards.

In [ ]:
total_rides = len(df)
successful = df['is_successful'].sum()
cancelled_driver = (df['booking_status'] == 'Cancelled by Driver').sum()
cancelled_customer = (df['booking_status'] == 'Cancelled by Customer').sum()
incomplete = (df['booking_status'] == 'Incomplete').sum()
total_non_success = total_rides - successful

completion_rate = round(successful / total_rides * 100, 2)
overall_cancel_rate = round((cancelled_driver + cancelled_customer) / total_rides * 100, 2)
driver_cancel_rate = round(cancelled_driver / total_rides * 100, 2)
customer_cancel_rate = round(cancelled_customer / total_rides * 100, 2)

total_revenue = round(df_success['booking_value'].sum(), 2)
avg_fare = round(df_success['booking_value'].mean(), 2)
avg_distance = round(df_success['ride_distance'].mean(), 2)
avg_vtat = round(df_success['avg_vtat'].mean(), 2)
avg_ctat = round(df_success['avg_ctat'].mean(), 2)
avg_driver_rating = round(df_success['driver_rating'].mean(), 2)
avg_customer_rating = round(df_success['customer_rating'].mean(), 2)

revenue_lost = round(total_non_success * avg_fare, 2)
revenue_recovery_15 = round(total_non_success * 0.15 * avg_fare, 2)
fleet_efficiency = round(successful / total_rides * 100, 2)

print('=== MASTER KPI SUMMARY ===')
print(f'Total Ride Requests       : {total_rides:,}')
print(f'Ride Completion Rate      : {completion_rate}%')
print(f'Overall Cancellation Rate : {overall_cancel_rate}%')
print(f'Driver Cancellation Rate  : {driver_cancel_rate}%')
print(f'Customer Cancellation Rate: {customer_cancel_rate}%')
print(f'Total Revenue Earned      : ₹{total_revenue:,.0f}')
print(f'Avg Fare per Ride         : ₹{avg_fare}')
print(f'Avg Ride Distance         : {avg_distance} km')
print(f'Avg VTAT                  : {avg_vtat} min')
print(f'Avg CTAT                  : {avg_ctat} min')
print(f'Avg Driver Rating         : {avg_driver_rating}')
print(f'Avg Customer Rating       : {avg_customer_rating}')
print(f'Estimated Revenue Lost    : ₹{revenue_lost:,.0f}')
print(f'Revenue Recovery (15%)    : ₹{revenue_recovery_15:,.0f}')
print(f'Fleet Efficiency          : {fleet_efficiency}%')

---
### Dashboard 1 — CEO Strategic Health
Preparing aggregations for: daily trend, booking status, top pickup locations, vehicle utilization, hourly demand, revenue comparison.

In [ ]:
# daily ride volume trend
daily_trend = df.groupby('date').agg(
    total_rides=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    cancelled=('is_cancelled', 'sum'),
    revenue=('booking_value', 'sum')
).reset_index()
daily_trend['completion_rate'] = (daily_trend['successful'] / daily_trend['total_rides'] * 100).round(2)
daily_trend['cancel_rate'] = (daily_trend['cancelled'] / daily_trend['total_rides'] * 100).round(2)
daily_trend['date'] = daily_trend['date'].dt.strftime('%Y-%m-%d')
print(f'daily trend: {len(daily_trend)} days')
daily_trend.head()

In [ ]:
# booking status breakdown
status_breakdown = df['booking_status'].value_counts().reset_index()
status_breakdown.columns = ['booking_status', 'count']
status_breakdown['pct'] = (status_breakdown['count'] / total_rides * 100).round(2)
print(status_breakdown.to_string(index=False))

In [ ]:
# top pickup locations by demand
top_locations = df.groupby('pickup_location').agg(
    total_requests=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    cancelled=('is_cancelled', 'sum'),
    revenue=('booking_value', 'sum')
).reset_index()
top_locations['completion_rate'] = (top_locations['successful'] / top_locations['total_requests'] * 100).round(2)
top_locations['cancel_rate'] = (top_locations['cancelled'] / top_locations['total_requests'] * 100).round(2)
top_locations['area_num'] = top_locations['pickup_location'].str.extract(r'(\d+)').astype(int)
top_locations = top_locations.sort_values('total_requests', ascending=False)
print(f'top 5 high demand areas:')
print(top_locations.head(5)[['pickup_location', 'total_requests', 'completion_rate']].to_string(index=False))

In [ ]:
# vehicle utilization
vehicle_util = df.groupby('vehicle_type').agg(
    total_bookings=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    cancelled=('is_cancelled', 'sum'),
    revenue_earned=('booking_value', 'sum'),
    avg_fare=('booking_value', 'mean'),
    avg_distance=('ride_distance', 'mean')
).reset_index()
vehicle_util['completion_rate'] = (vehicle_util['successful'] / vehicle_util['total_bookings'] * 100).round(2)
vehicle_util['cancel_rate'] = (vehicle_util['cancelled'] / vehicle_util['total_bookings'] * 100).round(2)
vehicle_util['revenue_lost'] = (vehicle_util['cancelled'] * avg_fare).round(2)
vehicle_util = vehicle_util.sort_values('total_bookings', ascending=False)
print(vehicle_util[['vehicle_type', 'total_bookings', 'completion_rate', 'cancel_rate', 'revenue_earned']].to_string(index=False))

In [ ]:
# hourly demand pattern
hourly_demand = df.groupby('hour').agg(
    total_rides=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    cancelled=('is_cancelled', 'sum'),
    revenue=('booking_value', 'sum')
).reset_index()
hourly_demand['completion_rate'] = (hourly_demand['successful'] / hourly_demand['total_rides'] * 100).round(2)
hourly_demand['cancel_rate'] = (hourly_demand['cancelled'] / hourly_demand['total_rides'] * 100).round(2)
print(f'hourly demand prepared: {len(hourly_demand)} hours')

In [ ]:
# master KPI card data for dashboard 1
d1_kpis = pd.DataFrame([{
    'total_ride_requests': total_rides,
    'ride_completion_rate_pct': completion_rate,
    'overall_cancellation_rate_pct': overall_cancel_rate,
    'total_revenue_inr': total_revenue,
    'revenue_lost_inr': revenue_lost,
    'revenue_recovery_15pct_inr': revenue_recovery_15
}])
print(d1_kpis.to_string(index=False))

---
### Dashboard 2 — Customer Experience
Preparing aggregations for: customer cancellation reasons, wait times, ratings, ride availability, vehicle preference, location issues.

In [ ]:
# customer cancellation reasons
cust_reasons = df[df['cancellation_party'] == 'Customer']['cancel_reason_customer'].value_counts().reset_index()
cust_reasons.columns = ['reason', 'count']
cust_reasons['pct'] = (cust_reasons['count'] / cancelled_customer * 100).round(2)
print('customer cancellation reasons:')
print(cust_reasons.to_string(index=False))

In [ ]:
# avg wait time and ratings by vehicle type
cust_experience = df_success.groupby('vehicle_type').agg(
    avg_ctat=('avg_ctat', 'mean'),
    avg_vtat=('avg_vtat', 'mean'),
    avg_driver_rating=('driver_rating', 'mean'),
    avg_customer_rating=('customer_rating', 'mean'),
    total_successful=('booking_id', 'count'),
    avg_fare=('booking_value', 'mean')
).reset_index().round(2)
print(cust_experience.to_string(index=False))

In [ ]:
# ride availability by time slot
availability = df.groupby('time_of_day').agg(
    total=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    customer_cancelled=('cancelled_by_customer', 'sum')
).reset_index()
availability['completion_rate'] = (availability['successful'] / availability['total'] * 100).round(2)
availability['customer_cancel_rate'] = (availability['customer_cancelled'] / availability['total'] * 100).round(2)
print(availability.to_string(index=False))

In [ ]:
# location wise ride availability issues
location_issues = df.groupby('pickup_location').agg(
    total=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    cancelled=('is_cancelled', 'sum'),
    incomplete=('incomplete_ride', 'sum'),
    avg_ctat=('avg_ctat', 'mean'),
    avg_customer_rating=('customer_rating', 'mean')
).reset_index().round(2)
location_issues['availability_issues'] = location_issues['cancelled'] + location_issues['incomplete']
location_issues['completion_rate'] = (location_issues['successful'] / location_issues['total'] * 100).round(2)
location_issues = location_issues.sort_values('availability_issues', ascending=False)
print('top 5 areas with worst availability:')
print(location_issues.head(5)[['pickup_location', 'total', 'availability_issues', 'completion_rate']].to_string(index=False))

In [ ]:
# payment method preference
payment_pref = df_success.groupby('payment_method').agg(
    count=('booking_id', 'count'),
    avg_fare=('booking_value', 'mean'),
    total_revenue=('booking_value', 'sum')
).reset_index().round(2)
payment_pref['pct'] = (payment_pref['count'] / len(df_success) * 100).round(2)
print(payment_pref.to_string(index=False))

In [ ]:
# KPI cards for dashboard 2
most_common_cust_reason = df[df['cancellation_party'] == 'Customer']['cancel_reason_customer'].value_counts().index[0]

d2_kpis = pd.DataFrame([{
    'avg_customer_rating': avg_customer_rating,
    'avg_driver_rating': avg_driver_rating,
    'customer_cancellation_rate_pct': customer_cancel_rate,
    'avg_ctat_minutes': avg_ctat,
    'most_common_customer_cancel_reason': most_common_cust_reason
}])
print(d2_kpis.to_string(index=False))

---
### Dashboard 3 — Finance & Operations
Preparing aggregations for: driver cancellation reasons, completed trips, fleet utilization, risk heatmap, location efficiency, revenue lost, distance distribution.

In [ ]:
# driver cancellation reasons
driver_reasons = df[df['cancellation_party'] == 'Driver']['cancel_reason_driver'].value_counts().reset_index()
driver_reasons.columns = ['reason', 'count']
driver_reasons['pct'] = (driver_reasons['count'] / cancelled_driver * 100).round(2)
print('driver cancellation reasons:')
print(driver_reasons.to_string(index=False))

In [ ]:
# fleet utilization by vehicle and time slot
fleet_util = df.groupby(['vehicle_type', 'time_of_day']).agg(
    total=('booking_id', 'count'),
    successful=('is_successful', 'sum'),
    driver_cancelled=('cancelled_by_driver', 'sum'),
    revenue=('booking_value', 'sum'),
    avg_vtat=('avg_vtat', 'mean'),
    avg_distance=('ride_distance', 'mean')
).reset_index().round(2)
fleet_util['completion_rate'] = (fleet_util['successful'] / fleet_util['total'] * 100).round(2)
fleet_util['driver_cancel_rate'] = (fleet_util['driver_cancelled'] / fleet_util['total'] * 100).round(2)
fleet_util['revenue_lost'] = (fleet_util['driver_cancelled'] * avg_fare).round(2)
print(f'fleet utilization table: {len(fleet_util)} combinations')

In [ ]:
# location wise completion efficiency
location_eff = df.groupby('pickup_location').agg(
    total_requested=('booking_id', 'count'),
    completed=('is_successful', 'sum'),
    driver_cancelled=('cancelled_by_driver', 'sum'),
    avg_vtat=('avg_vtat', 'mean'),
    revenue_earned=('booking_value', 'sum')
).reset_index().round(2)
location_eff['completion_efficiency'] = (location_eff['completed'] / location_eff['total_requested'] * 100).round(2)
location_eff['driver_cancel_rate'] = (location_eff['driver_cancelled'] / location_eff['total_requested'] * 100).round(2)
location_eff['revenue_lost'] = (location_eff['driver_cancelled'] * avg_fare).round(2)
location_eff['area_num'] = location_eff['pickup_location'].str.extract(r'(\d+)').astype(int)
location_eff = location_eff.sort_values('revenue_lost', ascending=False)
print('top 5 areas by revenue lost:')
print(location_eff.head(5)[['pickup_location', 'total_requested', 'completion_efficiency', 'revenue_lost']].to_string(index=False))

In [ ]:
# distance distribution buckets
df_success_copy = df_success.copy()
df_success_copy['distance_bucket'] = pd.cut(
    df_success_copy['ride_distance'],
    bins=[0, 5, 10, 20, 30, 50],
    labels=['0-5 km', '5-10 km', '10-20 km', '20-30 km', '30-50 km']
)
distance_dist = df_success_copy.groupby('distance_bucket').agg(
    count=('booking_id', 'count'),
    avg_fare=('booking_value', 'mean'),
    total_revenue=('booking_value', 'sum')
).reset_index().round(2)
distance_dist['pct'] = (distance_dist['count'] / len(df_success) * 100).round(2)
print(distance_dist.to_string(index=False))

In [ ]:
# KPI cards for dashboard 3
d3_kpis = pd.DataFrame([{
    'driver_cancellation_rate_pct': driver_cancel_rate,
    'avg_vtat_minutes': avg_vtat,
    'revenue_lost_driver_cancellations_inr': round(cancelled_driver * avg_fare, 2),
    'fleet_completion_efficiency_pct': fleet_efficiency,
    'revenue_recovery_15pct_inr': revenue_recovery_15
}])
print(d3_kpis.to_string(index=False))

### Saving all Tableau-ready files

In [ ]:
# Dashboard 1 — CEO
daily_trend.to_csv('/content/05_dashboard1_daily_trend.csv', index=False)
status_breakdown.to_csv('/content/05_dashboard1_status_breakdown.csv', index=False)
top_locations.to_csv('/content/05_dashboard1_locations.csv', index=False)
vehicle_util.to_csv('/content/05_dashboard1_vehicle_util.csv', index=False)
hourly_demand.to_csv('/content/05_dashboard1_hourly_demand.csv', index=False)
d1_kpis.to_csv('/content/05_dashboard1_kpis.csv', index=False)
print('Dashboard 1 files saved.')

# Dashboard 2 — Customer Experience
cust_reasons.to_csv('/content/05_dashboard2_cust_reasons.csv', index=False)
cust_experience.to_csv('/content/05_dashboard2_cust_experience.csv', index=False)
availability.to_csv('/content/05_dashboard2_availability.csv', index=False)
location_issues.to_csv('/content/05_dashboard2_location_issues.csv', index=False)
payment_pref.to_csv('/content/05_dashboard2_payment.csv', index=False)
d2_kpis.to_csv('/content/05_dashboard2_kpis.csv', index=False)
print('Dashboard 2 files saved.')

# Dashboard 3 — Finance & Operations
driver_reasons.to_csv('/content/05_dashboard3_driver_reasons.csv', index=False)
fleet_util.to_csv('/content/05_dashboard3_fleet_util.csv', index=False)
location_eff.to_csv('/content/05_dashboard3_location_efficiency.csv', index=False)
distance_dist.to_csv('/content/05_dashboard3_distance_dist.csv', index=False)
d3_kpis.to_csv('/content/05_dashboard3_kpis.csv', index=False)
print('Dashboard 3 files saved.')

# Master KPI summary
master_kpis = pd.DataFrame([{
    'total_ride_requests': total_rides,
    'ride_completion_rate_pct': completion_rate,
    'overall_cancellation_rate_pct': overall_cancel_rate,
    'driver_cancellation_rate_pct': driver_cancel_rate,
    'customer_cancellation_rate_pct': customer_cancel_rate,
    'total_revenue_inr': total_revenue,
    'avg_fare_inr': avg_fare,
    'avg_ride_distance_km': avg_distance,
    'avg_vtat_min': avg_vtat,
    'avg_ctat_min': avg_ctat,
    'avg_driver_rating': avg_driver_rating,
    'avg_customer_rating': avg_customer_rating,
    'revenue_lost_inr': revenue_lost,
    'revenue_recovery_15pct_inr': revenue_recovery_15,
    'fleet_efficiency_pct': fleet_efficiency
}])
master_kpis.to_csv('/content/05_master_kpis.csv', index=False)
print('Master KPI file saved.')
print()
print('All files ready for Tableau. Proceed to dashboard build.')

In [ ]:
# zipping all output files for easy download
import zipfile
import os

files = [
    '05_dashboard1_daily_trend.csv',
    '05_dashboard1_status_breakdown.csv',
    '05_dashboard1_locations.csv',
    '05_dashboard1_vehicle_util.csv',
    '05_dashboard1_hourly_demand.csv',
    '05_dashboard1_kpis.csv',
    '05_dashboard2_cust_reasons.csv',
    '05_dashboard2_cust_experience.csv',
    '05_dashboard2_availability.csv',
    '05_dashboard2_location_issues.csv',
    '05_dashboard2_payment.csv',
    '05_dashboard2_kpis.csv',
    '05_dashboard3_driver_reasons.csv',
    '05_dashboard3_fleet_util.csv',
    '05_dashboard3_location_efficiency.csv',
    '05_dashboard3_distance_dist.csv',
    '05_dashboard3_kpis.csv',
    '05_master_kpis.csv'
]

with zipfile.ZipFile('/content/notebook05_tableau_data.zip', 'w') as z:
    for f in files:
        if os.path.exists(f'/content/{f}'):
            z.write(f'/content/{f}', f)
            print(f'added: {f}')
        else:
            print(f'missing: {f}')

print('\ndownload notebook05_tableau_data.zip from the sidebar')

### What was prepared in this notebook

**Dashboard 1 — CEO (6 files)**
- Daily trend, status breakdown, top locations, vehicle utilization, hourly demand, KPI cards

**Dashboard 2 — Customer Experience (6 files)**
- Customer cancel reasons, experience by vehicle, ride availability, location issues, payment preference, KPI cards

**Dashboard 3 — Finance & Operations (5 files)**
- Driver cancel reasons, fleet utilization, location efficiency, distance distribution, KPI cards

**Master KPI file (1 file)**
- All 15 top-level KPIs in one row — use for KPI cards across all dashboards

Total: 18 CSV files ready for Tableau connection.